# RAG Pipeline: Data Ingestion থেকে Vector Database পর্যন্ত

এই নোটবুকে আমরা **RAG (Retrieval Augmented Generation)** pipeline-এর সবচেয়ে গুরুত্বপূর্ণ অংশ শিখব — ডেটা কিভাবে Vector Database-এ সংরক্ষণ করা হয়।

## পুরো RAG Pipeline:

```
Documents (PDF/CSV/Web/Text)
        ↓
   Document Loading          ← আগের নোটবুকে শিখেছি
        ↓
   Text Splitting            ← আজকের Part 1
  (Chunks তৈরি করা)
        ↓
   Embedding                 ← আজকের Part 2
  (Text → Vector)
        ↓
   Vector Store              ← আজকের Part 3
  (FAISS / ChromaDB)
        ↓
   Retrieval                 ← আজকের Part 4
  (Similarity Search)
        ↓
   LLM Generation            ← Anthropic Claude
```

## আজকের Topics:

| বিষয় | বিবরণ |
|---|---|
| **Text Splitting** | Documents-কে ছোট ছোট chunks-এ ভাগ করা |
| **Embeddings** | Text-কে mathematical vector-এ রূপান্তর করা |
| **FAISS** | Facebook-এর fast in-memory vector store |
| **ChromaDB** | Open-source persistent vector database |
| **Full Pipeline** | সব কিছু একসাথে — লোড → স্প্লিট → এম্বেড → সংরক্ষণ → রিট্রিভ → উত্তর |

---
## Step 1 — Environment Setup

`.env` ফাইল থেকে API key লোড করা এবং LLM তৈরি করা।

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY:", bool(ANTHROPIC_API_KEY))

ANTHROPIC_API_KEY: True


In [2]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)
print("LLM ready:", llm.model)

g:\all projects\AI agents code\agentic-course-krish-naik\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready: claude-haiku-4-5-20251001


---
## Step 2 — Documents লোড করা

আগের নোটবুকে শেখা `DirectoryLoader` ব্যবহার করে আমাদের sample text files লোড করব।
এই documents-গুলোই পরে split করা হবে।

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.document_loaders import PyPDFLoader

# Text files লোড করা
text_loader = DirectoryLoader(
    path="../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False,
)
text_docs = text_loader.load()

# PDF files লোড করা
pdf_loader = DirectoryLoader(
    path="../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=False,
)
pdf_docs = pdf_loader.load()

all_docs = text_docs + pdf_docs

print(f"Text documents: {len(text_docs)}")
print(f"PDF documents: {len(pdf_docs)}")
print(f"Total documents: {len(all_docs)}")
print()
for doc in all_docs:
    src = os.path.basename(doc.metadata.get("source", "?"))
    print(f"  [{src}] — {len(doc.page_content)} chars")

C:\Users\Nibras\AppData\Local\Temp\ipykernel_12132\2460817973.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Text documents: 3
PDF documents: 2
Total documents: 5

  [artificial_intelligence.txt] — 2182 chars
  [machine_learning.txt] — 2538 chars
  [natural_language_processing.txt] — 2810 chars
  [langchain_intro.pdf] — 1197 chars
  [rag_overview.pdf] — 1144 chars


---
# Part 1: Text Splitting — Documents-কে Chunks-এ ভাগ করা

## Text Splitting কেন দরকার?

LLM-এর **context window** সীমিত। একটি বড় PDF হয়তো ১০০ পেজের — এটা একবারে LLM-কে দেওয়া যাবে না।
তাছাড়া, RAG-এ আমরা শুধু **relevant অংশটুকু** retrieve করতে চাই, পুরো document নয়।

### সমস্যা:

```
বড় Document (1000 words)
→ LLM context limit exceed করবে
→ Irrelevant text থাকবে
→ Embedding quality কমে যাবে
```

### সমাধান:

```
বড় Document
→ Text Splitter
→ [Chunk 1 (200 words)] [Chunk 2 (200 words)] [Chunk 3 (200 words)] ...
→ প্রতিটি chunk আলাদাভাবে embed করা যাবে
→ Query অনুযায়ী শুধু relevant chunk retrieve করা যাবে
```

## দুটি গুরুত্বপূর্ণ Parameter:

| Parameter | মানে | Example |
|---|---|---|
| `chunk_size` | প্রতিটি chunk-এর সর্বোচ্চ size (characters) | `500` |
| `chunk_overlap` | দুটি পাশাপাশি chunk-এর মধ্যে shared content | `50` |

**Overlap কেন দরকার?** একটি বাক্য যদি দুটি chunk-এর boundary-তে পড়ে, overlap না থাকলে সেই context হারিয়ে যাবে।

```
Text:  [----chunk 1----][overlap][----chunk 2----][overlap][----chunk 3----]
```

---
## 1.1 CharacterTextSplitter

**`CharacterTextSplitter`** একটি নির্দিষ্ট character (default: `\n\n`) দিয়ে text split করে।

- **সহজ কিন্তু সীমাবদ্ধ** — শুধু একটি separator দিয়ে split করে
- যদি সেই separator না থাকে, তাহলে পুরো text একটি chunk হয়ে যায়
- `chunk_size` exceed করলে warning দেয় কিন্তু split করে না (যদি separator না পায়)

In [4]:
from langchain_text_splitters import CharacterTextSplitter

# Sample text
sample_text = """Artificial Intelligence is transforming the world.
It enables machines to learn from data and make decisions.

Machine Learning is a subset of AI.
It uses algorithms to find patterns in data without explicit programming.

Deep Learning uses neural networks with many layers.
It powers image recognition, speech processing, and language understanding.

Natural Language Processing helps computers understand human language.
It enables applications like chatbots, translation, and text analysis."""

# CharacterTextSplitter তৈরি করা
char_splitter = CharacterTextSplitter(
    separator="\n\n",     # এই character দিয়ে split করবে
    chunk_size=200,         # সর্বোচ্চ 200 character প্রতি chunk
    chunk_overlap=30,       # ৩০ character overlap
    length_function=len,    # character count ব্যবহার করে size measure করবে
)

chunks = char_splitter.split_text(sample_text)

print(f"Original text length: {len(sample_text)} characters")
print(f"Total chunks created: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

Original text length: 493 characters
Total chunks created: 4

--- Chunk 1 (109 chars) ---
Artificial Intelligence is transforming the world.
It enables machines to learn from data and make decisions.

--- Chunk 2 (109 chars) ---
Machine Learning is a subset of AI.
It uses algorithms to find patterns in data without explicit programming.

--- Chunk 3 (128 chars) ---
Deep Learning uses neural networks with many layers.
It powers image recognition, speech processing, and language understanding.

--- Chunk 4 (141 chars) ---
Natural Language Processing helps computers understand human language.
It enables applications like chatbots, translation, and text analysis.



---
## 1.2 RecursiveCharacterTextSplitter — সবচেয়ে ভালো পদ্ধতি ✅

**`RecursiveCharacterTextSplitter`** হল LangChain-এর **recommended default** text splitter।

এটি hierarchically একাধিক separator চেষ্টা করে:

```
1. প্রথমে: "\n\n"  (paragraph break)
2. তারপর: "\n"     (line break)
3. তারপর: " "       (space/word boundary)
4. শেষে:  ""        (character level)
```

**কেন ভালো?** এটি সবসময় চেষ্টা করে যতটা সম্ভব semantic unit (paragraph, sentence, word) ঠিক রাখতে।

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter তৈরি করা
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],  # এই order-এ try করবে
)

chunks_recursive = recursive_splitter.split_text(sample_text)

print(f"Original text length: {len(sample_text)} characters")
print(f"Total chunks: {len(chunks_recursive)}")
print()
for i, chunk in enumerate(chunks_recursive):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

Original text length: 493 characters
Total chunks: 2

--- Chunk 1 (220 chars) ---
Artificial Intelligence is transforming the world.
It enables machines to learn from data and make decisions.

Machine Learning is a subset of AI.
It uses algorithms to find patterns in data without explicit programming.

--- Chunk 2 (271 chars) ---
Deep Learning uses neural networks with many layers.
It powers image recognition, speech processing, and language understanding.

Natural Language Processing helps computers understand human language.
It enables applications like chatbots, translation, and text analysis.



In [6]:
# CharacterTextSplitter vs RecursiveCharacterTextSplitter তুলনা
print("=" * 55)
print("CharacterTextSplitter vs RecursiveCharacterTextSplitter")
print("=" * 55)

char_splitter_cmp = CharacterTextSplitter(
    separator="\n\n", chunk_size=300, chunk_overlap=50
)
recursive_splitter_cmp = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50
)

char_chunks = char_splitter_cmp.split_text(sample_text)
rec_chunks  = recursive_splitter_cmp.split_text(sample_text)

print(f"\nCharacterTextSplitter:          {len(char_chunks)} chunks")
print(f"RecursiveCharacterTextSplitter: {len(rec_chunks)} chunks")

print("\nChunk sizes comparison:")
print(f"  {'Chunk':<8} {'Char (chars)':<20} {'Recursive (chars)'}")
for i in range(max(len(char_chunks), len(rec_chunks))):
    c_size = len(char_chunks[i]) if i < len(char_chunks) else "-"
    r_size = len(rec_chunks[i]) if i < len(rec_chunks) else "-"
    print(f"  {i+1:<8} {str(c_size):<20} {r_size}")

CharacterTextSplitter vs RecursiveCharacterTextSplitter

CharacterTextSplitter:          2 chunks
RecursiveCharacterTextSplitter: 2 chunks

Chunk sizes comparison:
  Chunk    Char (chars)         Recursive (chars)
  1        220                  220
  2        271                  271


In [7]:
# chunk_overlap-এর প্রভাব বোঝা
print("Chunk Overlap-এর প্রভাব দেখা:")
print("=" * 50)

no_overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=0
)
with_overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=50
)

no_overlap_chunks  = no_overlap_splitter.split_text(sample_text)
with_overlap_chunks = with_overlap_splitter.split_text(sample_text)

print(f"\nchunk_overlap=0:  {len(no_overlap_chunks)} chunks")
print(f"chunk_overlap=50: {len(with_overlap_chunks)} chunks")

# প্রথম দুটি chunk-এর শেষ ও শুরু দেখা
print("\nOverlap ছাড়া:")
print(f"  Chunk 1 শেষ: ...{no_overlap_chunks[0][-60:]!r}")
print(f"  Chunk 2 শুরু: {no_overlap_chunks[1][:60]!r}...")

print("\nOverlap সহ:")
print(f"  Chunk 1 শেষ: ...{with_overlap_chunks[0][-60:]!r}")
print(f"  Chunk 2 শুরু: {with_overlap_chunks[1][:60]!r}...")
print("\n→ দেখো chunk 2-এর শুরু chunk 1-এর শেষের সাথে overlap করছে!")

Chunk Overlap-এর প্রভাব দেখা:

chunk_overlap=0:  4 chunks
chunk_overlap=50: 4 chunks

Overlap ছাড়া:
  Chunk 1 শেষ: ...'.\nIt enables machines to learn from data and make decisions.'
  Chunk 2 শুরু: 'Machine Learning is a subset of AI.\nIt uses algorithms to fi'...

Overlap সহ:
  Chunk 1 শেষ: ...'.\nIt enables machines to learn from data and make decisions.'
  Chunk 2 শুরু: 'Machine Learning is a subset of AI.\nIt uses algorithms to fi'...

→ দেখো chunk 2-এর শুরু chunk 1-এর শেষের সাথে overlap করছে!


---
## 1.3 `split_documents()` — Document Objects Split করা

`split_text()` শুধু string নিয়ে কাজ করে। কিন্তু Document Loader থেকে পাওয়া **`Document` objects** split করতে `split_documents()` ব্যবহার করতে হয়।

**গুরুত্বপূর্ণ:** `split_documents()` প্রতিটি chunk-এ original document-এর **metadata automatically copy** করে।

In [8]:
# Real documents split করা
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)

# Document objects split করা
split_docs = splitter.split_documents(all_docs)

print(f"Original documents: {len(all_docs)}")
print(f"Total chunks after splitting: {len(split_docs)}")
print()

# প্রতিটি source থেকে কতটি chunk হয়েছে দেখা
from collections import Counter
source_counts = Counter(
    os.path.basename(d.metadata.get("source", "?")) for d in split_docs
)
print("Chunks per source file:")
for src, count in source_counts.items():
    print(f"  {src}: {count} chunks")

print()
print("=== প্রথম chunk-এর উদাহরণ ===")
first_chunk = split_docs[0]
print(f"Content ({len(first_chunk.page_content)} chars):")
print(first_chunk.page_content[:300])
print(f"\nMetadata: {first_chunk.metadata}")

Original documents: 5
Total chunks after splitting: 28

Chunks per source file:
  artificial_intelligence.txt: 7 chunks
  machine_learning.txt: 7 chunks
  natural_language_processing.txt: 8 chunks
  langchain_intro.pdf: 3 chunks
  rag_overview.pdf: 3 chunks

=== প্রথম chunk-এর উদাহরণ ===
Content (312 chars):
Artificial Intelligence: An Overview

Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of performing tasks that typically require human intelligence. These tasks include understanding natural language, recognizing patterns, solving problems, and makin

Metadata: {'source': '..\\data\\text_files\\artificial_intelligence.txt'}


In [9]:
# Chunk size distribution দেখা
chunk_sizes = [len(d.page_content) for d in split_docs]
print(f"Chunk size statistics:")
print(f"  Total chunks   : {len(chunk_sizes)}")
print(f"  Min size       : {min(chunk_sizes)} chars")
print(f"  Max size       : {max(chunk_sizes)} chars")
print(f"  Average size   : {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")

# Size distribution
small  = sum(1 for s in chunk_sizes if s < 200)
medium = sum(1 for s in chunk_sizes if 200 <= s < 400)
large  = sum(1 for s in chunk_sizes if s >= 400)
print(f"\nDistribution:")
print(f"  Small  (<200 chars) : {small} chunks")
print(f"  Medium (200-400)    : {medium} chunks")
print(f"  Large  (400+ chars) : {large} chunks")

Chunk size statistics:
  Total chunks   : 28
  Min size       : 149 chars
  Max size       : 498 chars
  Average size   : 362 chars

Distribution:
  Small  (<200 chars) : 1 chunks
  Medium (200-400)    : 13 chunks
  Large  (400+ chars) : 14 chunks


---
# Part 2: Embeddings — Text-কে Vector-এ রূপান্তর করা

## Embedding কী?

**Embedding** হল text-কে একটি numerical vector (সংখ্যার array) এ রূপান্তর করার পদ্ধতি।

```
"মেশিন লার্নিং হল AI-এর একটি শাখা"
        ↓  Embedding Model
[0.23, -0.15, 0.87, 0.42, -0.63, ...] ← 384-dimensional vector
```

### কেন Vector?

| Text | Vector |
|---|---|
| Computer সরাসরি text বোঝে না | Computer সহজেই numbers process করতে পারে |
| "similar" মানে কী? — সংজ্ঞায়িত করা কঠিন | Vector space-এ "similar" = vectors কাছাকাছি |
| Search করা কঠিন | Vector math দিয়ে fast similarity search |

### Semantic Similarity:

Embedding-এর সবচেয়ে গুরুত্বপূর্ণ বৈশিষ্ট্য হল **semantically similar** text-গুলোর vectors **কাছাকাছি** থাকে।

```
"AI is transforming healthcare"     → [0.23, 0.45, ...]
"Artificial Intelligence in medicine" → [0.21, 0.43, ...]  ← কাছাকাছি!
"The weather is sunny today"        → [-0.52, 0.12, ...]  ← দূরে!
```

## আমরা ব্যবহার করব: `sentence-transformers`

`sentence-transformers` library-র `all-MiniLM-L6-v2` model:
- **Fast** এবং **lightweight** (CPU-তেও ভালো চলে)
- **384-dimensional** vectors তৈরি করে
- Open-source, **API key দরকার নেই**
- LangChain-এ `HuggingFaceEmbeddings` class দিয়ে ব্যবহার করা যায়

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# Embedding model লোড করা (প্রথমবার download হবে)
print("Embedding model লোড হচ্ছে...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)
print("Model লোড সম্পন্ন!")

# একটি text embed করা
sample = "Machine Learning is a subset of Artificial Intelligence."
vector = embeddings.embed_query(sample)

print(f"\nText: {sample!r}")
print(f"Vector dimension: {len(vector)}")
print(f"First 10 values: {[round(v, 4) for v in vector[:10]]}")
print(f"Vector type: {type(vector)}")

Embedding model লোড হচ্ছে...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2862.30it/s]


Model লোড সম্পন্ন!

Text: 'Machine Learning is a subset of Artificial Intelligence.'
Vector dimension: 384
First 10 values: [-0.0235, -0.0106, 0.0719, 0.0304, 0.0286, -0.0388, -0.0282, -0.0163, -0.0576, -0.0034]
Vector type: <class 'list'>


In [11]:
# embed_query vs embed_documents
print("embed_query vs embed_documents:")
print("=" * 45)

# embed_query — একটি text (str input, list output)
query = "What is deep learning?"
q_vec = embeddings.embed_query(query)
print(f"embed_query:")
print(f"  Input: {query!r}")
print(f"  Output type: list of {len(q_vec)} floats")

# embed_documents — একাধিক text (list input, list of lists output)
docs_texts = [
    "Deep learning uses neural networks.",
    "Machine learning finds patterns in data.",
    "The sky is blue and the sun is bright.",
]
d_vecs = embeddings.embed_documents(docs_texts)
print(f"\nembed_documents:")
print(f"  Input: {len(docs_texts)} texts")
print(f"  Output: {len(d_vecs)} vectors, each {len(d_vecs[0])} dimensions")

embed_query vs embed_documents:
embed_query:
  Input: 'What is deep learning?'
  Output type: list of 384 floats

embed_documents:
  Input: 3 texts
  Output: 3 vectors, each 384 dimensions


In [12]:
import math

def cosine_similarity(v1, v2):
    dot = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a * a for a in v1))
    mag2 = math.sqrt(sum(b * b for b in v2))
    return dot / (mag1 * mag2) if (mag1 and mag2) else 0.0

# Semantic similarity demo
sentences = [
    "Artificial Intelligence is transforming technology.",
    "AI and machine learning are changing the tech industry.",
    "The stock market fell sharply today.",
    "Deep learning models require large datasets.",
]

query_sent = "What is Artificial Intelligence?"
query_vec = embeddings.embed_query(query_sent)
sent_vecs = embeddings.embed_documents(sentences)

print(f'Query: "{query_sent}"')
print()
print("Similarity scores (higher = more similar):")
print("-" * 60)
scores = [(sentences[i], cosine_similarity(query_vec, sent_vecs[i]))
          for i in range(len(sentences))]
scores.sort(key=lambda x: x[1], reverse=True)
for sent, score in scores:
    bar = "█" * int(score * 20)
    print(f"  {score:.4f} {bar}")
    print(f"  {sent!r}")
    print()

Query: "What is Artificial Intelligence?"

Similarity scores (higher = more similar):
------------------------------------------------------------
  0.6286 ████████████
  'Artificial Intelligence is transforming technology.'

  0.4649 █████████
  'AI and machine learning are changing the tech industry.'

  0.2025 ████
  'Deep learning models require large datasets.'

  0.0386 
  'The stock market fell sharply today.'



---
# Part 3: FAISS — Fast In-Memory Vector Store

## FAISS কী?

**FAISS** (Facebook AI Similarity Search) হল Facebook Research-এর তৈরি একটি library যা **millions of vectors**-এর মধ্যে দ্রুত similarity search করতে পারে।

### বৈশিষ্ট্য:

| বৈশিষ্ট্য | বিবরণ |
|---|---|
| **Fast** | Billions of vectors-এ millisecond-level search |
| **In-Memory** | RAM-এ রাখে — persistent নয় (save করা যায়) |
| **Open-source** | Facebook AI Research |
| **No setup** | Server দরকার নেই |

### কখন ব্যবহার করবে?
- **Development/Prototyping** — দ্রুত শুরু করতে
- **Small to medium datasets** — লক্ষাধিক vectors পর্যন্ত
- **No persistence needed** — প্রতিবার নতুন করে তৈরি করা ঠিক আছে

### Pipeline:

```
Documents → split_documents() → chunks
chunks + embeddings → FAISS.from_documents() → vector_store
query → embeddings.embed_query() → query_vector
query_vector → vector_store.similarity_search() → relevant chunks
```

In [13]:
from langchain_community.vectorstores import FAISS

print("FAISS Vector Store তৈরি হচ্ছে...")
print(f"Input: {len(split_docs)} document chunks")

# from_documents: chunks embed করে FAISS index-এ রাখে
faiss_store = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings,
)

print(f"\nFAISS index তৈরি সম্পন্ন!")
print(f"Total vectors indexed: {faiss_store.index.ntotal}")

FAISS Vector Store তৈরি হচ্ছে...
Input: 28 document chunks

FAISS index তৈরি সম্পন্ন!
Total vectors indexed: 28


In [14]:
# Similarity Search — প্রশ্ন করে relevant chunks খোঁজা
query = "What is machine learning and how does it work?"

results = faiss_store.similarity_search(
    query=query,
    k=3,           # top 3 most similar chunks
)

print(f'Query: "{query}"')
print(f"Top {len(results)} relevant chunks:")
print("=" * 60)

for i, doc in enumerate(results):
    src = os.path.basename(doc.metadata.get("source", "?"))
    print(f"\n[{i+1}] Source: {src}")
    print(f"Content: {doc.page_content[:300]}")
    print("-" * 40)

Query: "What is machine learning and how does it work?"
Top 3 relevant chunks:

[1] Source: machine_learning.txt
Content: Machine Learning: A Comprehensive Guide

Machine Learning (ML) is a field of artificial intelligence that enables computers to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and use it to learn for themselves.

Core Con
----------------------------------------

[2] Source: artificial_intelligence.txt
Content: Machine Learning
Machine Learning (ML) is a subset of AI that enables computers to learn from data without being explicitly programmed. Key ML techniques include:
- Supervised Learning: Training on labeled data
- Unsupervised Learning: Finding patterns in unlabeled data
- Reinforcement Learning: Lea
----------------------------------------

[3] Source: machine_learning.txt
Content: Model Training
During training, the algorithm adjusts its internal parameters to minimize prediction 

In [15]:
# similarity_search_with_score — score সহ results
results_with_score = faiss_store.similarity_search_with_score(
    query="What are the types of neural networks?",
    k=3,
)

print("Score সহ similarity search:")
print("(FAISS score: lower = more similar, L2 distance)")
print("=" * 55)
for doc, score in results_with_score:
    src = os.path.basename(doc.metadata.get("source", "?"))
    print(f"\nScore: {score:.4f} | Source: {src}")
    print(f"Content: {doc.page_content[:200]}")

Score সহ similarity search:
(FAISS score: lower = more similar, L2 distance)

Score: 0.7744 | Source: machine_learning.txt
Content: Types of Machine Learning

1. Supervised Learning
The model learns from labeled training data. Common algorithms:
- Linear Regression: Predicts continuous values
- Logistic Regression: Classifies into

Score: 0.9403 | Source: artificial_intelligence.txt
Content: Types of AI
There are several types of AI based on capability:
1. Narrow AI (Weak AI): Designed for specific tasks, like image recognition or language translation. Examples include Siri, Alexa, and ch

Score: 1.0277 | Source: artificial_intelligence.txt
Content: Deep Learning
Deep Learning uses neural networks with many layers (hence "deep") to process complex data. It powers modern breakthroughs in image recognition, speech processing, and natural language u


In [16]:
# as_retriever — LangChain chain-এ ব্যবহারের জন্য
retriever = faiss_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

# retriever দিয়ে query করা
query = "Explain reinforcement learning"
retrieved = retriever.invoke(query)

print(f'Retriever query: "{query}"')
print(f"Retrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved):
    src = os.path.basename(doc.metadata.get("source", "?"))
    print(f"  [{i+1}] {src}: {doc.page_content[:150]}...")

Retriever query: "Explain reinforcement learning"
Retrieved 2 documents:
  [1] artificial_intelligence.txt: Machine Learning
Machine Learning (ML) is a subset of AI that enables computers to learn from data without being explicitly programmed. Key ML techniq...
  [2] machine_learning.txt: 2. Unsupervised Learning
The model finds hidden patterns in unlabeled data. Common algorithms:
- K-Means Clustering: Groups similar data points
- Prin...


In [17]:
# FAISS save এবং load করা
import os

save_path = "../data/faiss_index"

# Save করা
faiss_store.save_local(save_path)
print(f"FAISS index সংরক্ষিত: {save_path}/")

# Load করা (নতুন session-এ এভাবে load করা যাবে)
loaded_faiss = FAISS.load_local(
    folder_path=save_path,
    embeddings=embeddings,
    allow_dangerous_deserialization=True,  # locally saved index-এর জন্য
)

print(f"FAISS index লোড সম্পন্ন! Vectors: {loaded_faiss.index.ntotal}")

# Verify: loaded store দিয়েও search করা যাচ্ছে
test_results = loaded_faiss.similarity_search("deep learning neural networks", k=2)
print(f"Loaded index থেকে search: {len(test_results)} results পাওয়া গেছে ✓")

FAISS index সংরক্ষিত: ../data/faiss_index/
FAISS index লোড সম্পন্ন! Vectors: 28
Loaded index থেকে search: 2 results পাওয়া গেছে ✓


---
# Part 4: ChromaDB — Persistent Vector Database

## ChromaDB কী?

**ChromaDB** হল একটি open-source **vector database** যা disk-এ data persist করে।

### FAISS vs ChromaDB:

| Feature | FAISS | ChromaDB |
|---|---|---|
| **Storage** | In-memory (RAM) | Disk (persistent) |
| **Speed** | অত্যন্ত দ্রুত | দ্রুত |
| **Persistence** | save/load দরকার | স্বয়ংক্রিয় |
| **Setup** | কোনো server নেই | Embedded mode (server ছাড়া) |
| **Filtering** | সীমিত | metadata filter সহজ |
| **Scale** | লক্ষ vectors পর্যন্ত | লক্ষ থেকে কোটি |

### কখন ChromaDB ব্যবহার করবে?
- **Production apps** — data persist রাখতে হবে
- **Metadata filtering** — specific source বা date filter করতে হবে
- **Larger datasets** — FAISS-এর RAM limit ছাড়িয়ে গেলে

In [19]:
import chromadb
from langchain_chroma import Chroma

chroma_path = "../data/chroma_db"

# ChromaDB vector store তৈরি করা
print("ChromaDB Vector Store তৈরি হচ্ছে...")

chroma_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=chroma_path,   # এখানে disk-এ save হবে
    collection_name="rag_documents", # collection নাম
)

print(f"ChromaDB তৈরি সম্পন্ন!")
print(f"Persist directory: {chroma_path}")
print(f"Collection: rag_documents")
print(f"Total documents: {chroma_store._collection.count()}")

ChromaDB Vector Store তৈরি হচ্ছে...
ChromaDB তৈরি সম্পন্ন!
Persist directory: ../data/chroma_db
Collection: rag_documents
Total documents: 28


In [20]:
# ChromaDB similarity search
query = "What are applications of artificial intelligence?"

chroma_results = chroma_store.similarity_search(
    query=query,
    k=3,
)

print(f'Query: "{query}"')
print(f"Top {len(chroma_results)} results:")
print("=" * 60)

for i, doc in enumerate(chroma_results):
    src = os.path.basename(doc.metadata.get("source", "?"))
    print(f"\n[{i+1}] Source: {src}")
    print(f"Content: {doc.page_content[:300]}")

Query: "What are applications of artificial intelligence?"
Top 3 results:

[1] Source: artificial_intelligence.txt
Content: Artificial Intelligence: An Overview

Artificial Intelligence (AI) is a branch of computer science that aims to create machines capable of performing tasks that typically require human intelligence. These tasks include understanding natural language, recognizing patterns, solving problems, and makin

[2] Source: artificial_intelligence.txt
Content: Applications of AI
AI is transforming numerous industries:
- Healthcare: Disease diagnosis, drug discovery, medical imaging
- Finance: Fraud detection, algorithmic trading, credit scoring
- Transportation: Self-driving cars, route optimization
- Education: Personalized learning, intelligent tutoring

[3] Source: artificial_intelligence.txt
Content: Types of AI
There are several types of AI based on capability:
1. Narrow AI (Weak AI): Designed for specific tasks, like image recognition or language translation. Examples i

In [21]:
# Chroma - similarity search with relevance score (0 থেকে 1)
results_scored = chroma_store.similarity_search_with_relevance_scores(
    query="supervised vs unsupervised learning",
    k=4,
)

print("Relevance scores সহ Chroma search:")
print("(score: 1.0 = সবচেয়ে relevant, 0.0 = irrelevant)")
print("=" * 55)

for doc, score in results_scored:
    src = os.path.basename(doc.metadata.get("source", "?"))
    bar = "█" * int(score * 20)
    print(f"\nScore: {score:.4f} {bar}")
    print(f"Source: {src}")
    print(f"Content: {doc.page_content[:200]}")

Relevance scores সহ Chroma search:
(score: 1.0 = সবচেয়ে relevant, 0.0 = irrelevant)

Score: 0.4164 ████████
Source: machine_learning.txt
Content: 2. Unsupervised Learning
The model finds hidden patterns in unlabeled data. Common algorithms:
- K-Means Clustering: Groups similar data points
- Principal Component Analysis: Reduces data dimensions


Score: 0.3014 ██████
Source: artificial_intelligence.txt
Content: Machine Learning
Machine Learning (ML) is a subset of AI that enables computers to learn from data without being explicitly programmed. Key ML techniques include:
- Supervised Learning: Training on la

Score: 0.1917 ███
Source: machine_learning.txt
Content: Types of Machine Learning

1. Supervised Learning
The model learns from labeled training data. Common algorithms:
- Linear Regression: Predicts continuous values
- Logistic Regression: Classifies into

Score: 0.1422 ██
Source: machine_learning.txt
Content: Core Concepts

Training Data
Machine learning models learn from traini

In [22]:
# Chroma load করা (existing database থেকে)
existing_chroma = Chroma(
    persist_directory=chroma_path,
    embedding_function=embeddings,
    collection_name="rag_documents",
)

print(f"Existing ChromaDB লোড সম্পন্ন!")
print(f"Total documents: {existing_chroma._collection.count()}")

# Retriever তৈরি করা
chroma_retriever = existing_chroma.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

test_query = "transformer models in NLP"
retrieved = chroma_retriever.invoke(test_query)
print(f"\nRetriever query: {test_query!r}")
print(f"Retrieved: {len(retrieved)} documents")

Existing ChromaDB লোড সম্পন্ন!
Total documents: 28

Retriever query: 'transformer models in NLP'
Retrieved: 3 documents


---
# Part 5: Full RAG Pipeline — সব কিছু একসাথে

এখন আমরা পুরো **data ingestion pipeline** একসাথে দেখব:

```
1. Document Loading    →  DirectoryLoader / PyPDFLoader
2. Text Splitting      →  RecursiveCharacterTextSplitter
3. Embedding           →  HuggingFaceEmbeddings (sentence-transformers)
4. Vector Store        →  FAISS / ChromaDB
5. Retrieval           →  similarity_search
6. LLM Generation      →  Claude (Anthropic)
```

এই pattern-টি **যেকোনো RAG application**-এর ভিত্তি।

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ─── Step 1: Prompt Template ───────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_template("""
তুমি একজন সহায়ক AI assistant। নিচের context ব্যবহার করে প্রশ্নের উত্তর দাও।
যদি context-এ উত্তর না থাকে, তাহলে বলো "আমি এই বিষয়ে context পাইনি।"
উত্তর বাংলায় দাও।

Context:
{context}

প্রশ্ন: {question}

উত্তর:
""")

def rag_answer(question, vector_store, k=3):
    # Step 2: Retrieve relevant chunks
    relevant_docs = vector_store.similarity_search(question, k=k)

    # Step 3: Context তৈরি করা
    context = "\n\n".join([
        f"[Source: {os.path.basename(d.metadata.get('source', '?'))}]\n{d.page_content}"
        for d in relevant_docs
    ])

    # Step 4: LLM দিয়ে উত্তর তৈরি করা
    chain = rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    return answer, relevant_docs

# Test করা
question = "Machine Learning কী এবং এর কত ধরনের পদ্ধতি আছে?"
answer, docs = rag_answer(question, faiss_store)

print(f"প্রশ্ন: {question}")
print(f"\nRetrieved sources:")
for d in docs:
    print(f"  - {os.path.basename(d.metadata.get('source', '?'))}")
print(f"\nউত্তর:")
print(answer)

প্রশ্ন: Machine Learning কী এবং এর কত ধরনের পদ্ধতি আছে?

Retrieved sources:
  - machine_learning.txt
  - machine_learning.txt
  - natural_language_processing.txt

উত্তর:
# Machine Learning সম্পর্কে

## Machine Learning কী?

Machine Learning (ML) হল কৃত্রিম বুদ্ধিমত্তার একটি ক্ষেত্র যা কম্পিউটারকে অভিজ্ঞতা থেকে শিখতে এবং উন্নত হতে সক্ষম করে। এটি এমন অ্যালগরিদম তৈরিতে ফোকাস করে যা ডেটা অ্যাক্সেস করতে এবং নিজেরা শিখতে পারে, বিনা স্পষ্ট প্রোগ্রামিং এর।

## ML-এর পদ্ধতি সম্পর্কে

প্রদত্ত context-এ Machine Learning-এর ধরনগুলির বিস্তারিত তথ্য দেওয়া নেই। তবে context-এ **কিছু গুরুত্বপূর্ণ ধারণা** উল্লেখ আছে:

- **Overfitting এবং Underfitting**: এই দুটি মূল সমস্যা যা ML মডেল তৈরির সময় মোকাবেলা করতে হয়
- **নিয়মিতকরণ (Regularization)**: Overfitting রোধের কৌশল

ML-এর বিভিন্ন ধরনের পদ্ধতি (যেমন - Supervised Learning, Unsupervised Learning, Reinforcement Learning ইত্যাদি) সম্পর্কে আমি context-এ বিস্তারিত তথ্য পাইনি।


In [24]:
# আরও কয়েকটি প্রশ্ন করা
questions = [
    "Natural Language Processing-এ Transformer model কীভাবে কাজ করে?",
    "Deep Learning এবং Machine Learning-এর মধ্যে পার্থক্য কী?",
    "AI-এর ভবিষ্যৎ সম্পর্কে কী বলা হয়েছে?",
]

for q in questions:
    print("=" * 60)
    print(f"প্রশ্ন: {q}")
    answer, docs = rag_answer(q, chroma_store, k=2)
    sources = [os.path.basename(d.metadata.get("source", "?")) for d in docs]
    print(f"Sources: {sources}")
    print(f"উত্তর: {answer[:400]}")
    print()

প্রশ্ন: Natural Language Processing-এ Transformer model কীভাবে কাজ করে?
Sources: ['natural_language_processing.txt', 'natural_language_processing.txt']
উত্তর: আমি এই বিষয়ে context পাইনি।

Context-এ Transformer model কীভাবে কাজ করে তার বিস্তারিত প্রক্রিয়া সম্পর্কে তথ্য নেই। তবে Context থেকে জানা যায় যে:

- Transformer architecture NLP-তে বিপ্লব এনেছিল "Attention is All You Need" paper (2017) এর মাধ্যমে।
- এর উপর ভিত্তি করে তৈরি হয়েছে BERT, GPT series, T5, Claude এবং LLaMA এর মতো প্রধান মডেলগুলো।
- এই মডেলগুলো Machine Translation, Text Summarization এ

প্রশ্ন: Deep Learning এবং Machine Learning-এর মধ্যে পার্থক্য কী?
Sources: ['artificial_intelligence.txt', 'machine_learning.txt']
উত্তর: আমি এই বিষয়ে context পাইনি।

Context-এ Deep Learning এবং Machine Learning-এর মধ্যে পার্থক্য সম্পর্কে সরাসরি তুলনা করা হয়নি। Context-এ শুধুমাত্র Deep Learning এবং Machine Learning সম্পর্কে আলাদা আলাদা তথ্য রয়েছে, কিন্তু এই দুটির মধ্যে পার্থক্য ব্যাখ্যা করা হয়নি।

প্রশ্ন: AI-এর ভবিষ্যৎ সম্পর্কে কী ব

In [25]:
# Complete pipeline function — পুরো flow এক জায়গায়
def build_rag_pipeline(data_dir, chunk_size=500, chunk_overlap=80):
    """
    Full RAG ingestion pipeline:
      data_dir → load → split → embed → FAISS store
    """
    # 1. Load
    loader = DirectoryLoader(
        path=data_dir,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        show_progress=False,
    )
    docs = loader.load()

    # 2. Split
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)

    # 3. Embed + Store
    embed_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    store = FAISS.from_documents(chunks, embed_model)

    return store, chunks

# Pipeline চালানো
store, chunks = build_rag_pipeline("../data/text_files")
print(f"Pipeline complete!")
print(f"  Chunks: {len(chunks)}")
print(f"  FAISS vectors: {store.index.ntotal}")

# Quick test
res = store.similarity_search("what is reinforcement learning?", k=1)
print(f"  Test query result: {res[0].page_content[:150]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7365.33it/s]


Pipeline complete!
  Chunks: 22
  FAISS vectors: 22
  Test query result: Machine Learning
Machine Learning (ML) is a subset of AI that enables computers to learn from data without being explicitly programmed. Key ML techniq...


---
## সারসংক্ষেপ

### এই নোটবুকে আমরা যা শিখলাম:

#### Text Splitting:
| Splitter | কখন ব্যবহার | Key Params |
|---|---|---|
| `CharacterTextSplitter` | Simple splitting, নির্দিষ্ট separator | `separator`, `chunk_size`, `chunk_overlap` |
| `RecursiveCharacterTextSplitter` | **Default choice** ✅ | `chunk_size`, `chunk_overlap`, `separators` |

#### Embedding:
| Class | Model | Package |
|---|---|---|
| `HuggingFaceEmbeddings` | `all-MiniLM-L6-v2` (384-dim) | `langchain-huggingface` |

#### Vector Stores:
| Store | Storage | কখন ব্যবহার |
|---|---|---|
| **FAISS** | In-memory (save করা যায়) | Dev/prototype, fast search |
| **ChromaDB** | Disk (automatic persist) | Production, metadata filter |

### পরবর্তী ধাপ:
- **Retrieval Chain** — prompt + retriever + LLM একসাথে LCEL দিয়ে
- **Advanced Retrieval** — MMR (Maximal Marginal Relevance), parent document retriever
- **Hybrid Search** — keyword + semantic search একসাথে